In [ ]:
# %%
# Installation des bibliothèques nécessaires :
# - pandas : manipulation et analyse de données tabulaires
# - minio : client Python pour interagir avec un stockage objet MinIO (compatible S3)
 
%pip install pandas  minio

/workspaces/TODO/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import io           # Pour manipuler des flux de bytes en mémoire (évite d'écrire sur disque)
import pandas as pd # Bibliothèque principale pour la manipulation de DataFrames
import os           # Pour accéder aux variables d'environnement (credentials MinIO)

from minio import Minio

# --- Connexion au serveur MinIO ---
# MinIO est un système de stockage objet compatible S3, souvent utilisé en local ou on-premise.
# Les credentials sont lus depuis les variables d'environnement pour ne pas les écrire en dur.
client = Minio(
    "localhost:9000",                          # Adresse du serveur MinIO (ici en local)
    access_key=os.getenv("MINIO_ROOT_USER"),   # Identifiant d'accès (variable d'env)
    secret_key=os.getenv("MINIO_ROOT_PASSWORD"),# Mot de passe (variable d'env)
    secure=False,                              # Pas de TLS (connexion HTTP, approprié en local)
)

# --- Chargement du fichier des effectifs départementaux ---
# get_object récupère un fichier depuis le bucket "datasets" au chemin "raw/effectifs.csv"
response = client.get_object(
    "datasets",         # Nom du bucket MinIO
    "raw/effectifs.csv",# Chemin de l'objet dans le bucket
)

data = response.read()         # Lecture complète du contenu binaire de la réponse
response.close()               # Fermeture de la connexion HTTP
response.release_conn()        # Libération de la connexion dans le pool (bonne pratique)

# Conversion des bytes en DataFrame pandas
# io.BytesIO transforme les bytes bruts en un "fichier virtuel" lisible par read_csv
# sep=';' car le CSV utilise le point-virgule comme séparateur (format français courant)
df_dep = pd.read_csv(io.BytesIO(data), sep=';')
df_dep.head() # Aperçu des 5 premières lignes pour vérification

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop,Npop,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri
0,2019,"Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...",POP_TOT_ABS,45-49,9,76,11,"14,150",24090,59,"1,2,3",de 45 à 49 ans,tous sexes,17
1,2019,"Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...",POP_TOT_ABS,45-49,9,76,12,"10,400",17070,61,"1,2,3",de 45 à 49 ans,tous sexes,17
2,2019,"Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...",POP_TOT_ABS,45-49,9,76,30,"30,040",49750,60,"1,2,3",de 45 à 49 ans,tous sexes,17
3,2019,"Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...",POP_TOT_ABS,45-49,9,76,31,"57,210",93080,61,"1,2,3",de 45 à 49 ans,tous sexes,17
4,2019,"Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...","Pas de pathologie repérée, traitement, materni...",POP_TOT_ABS,45-49,9,76,32,"7,490",12260,61,"1,2,3",de 45 à 49 ans,tous sexes,17


In [ ]:
# --- Chargement du fichier des dépenses nationales ---
# Même logique que ci-dessus, mais pour le fichier agrégé au niveau national
response2 = client.get_object(
    "datasets",
    "raw/depenses.csv",
)

data = response2.read()
response2.close()
response2.release_conn()

df_nat = pd.read_csv(io.BytesIO(data), sep=';')
df_nat.head()

,annee,patho_niv1,patho_niv2,patho_niv3,top,dep_niv_1,dep_niv_2,montant,Ntop,N_recourant_au_poste,montant_moy,Niveau prioritaire,tri,type_somme
0,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Actes et consultations externes MCO secteur pu...,120435276,"1,690,110","974,410",71,"1,2,3",16,Partiel
1,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations en psychiatrie secteur privé ...,0,"1,690,110","6,500",0,"1,2,3",16,Partiel
2,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Hospitalisations (tous secteurs),Hospitalisations liste en sus MCO secteur publ...,43781491,"1,690,110","66,920",26,"1,2,3",16,Partiel
3,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Indemnités journalières maladie et AT/MP rembo...,86844655,"1,690,110","104,110",51,"1,2,3",16,Partiel
4,2017,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,Prestations en espèces,Total prestations en espèces remboursées,361210342,"1,690,110","208,230",214,"1,2,3",16,Total


In [ ]:
# --- Renommage des colonnes pour lever l'ambiguïté lors de la fusion ---
# Les deux fichiers contiennent une colonne "Ntop" (nombre de patients traités "top").
# On les distingue explicitement avant tout croisement des deux DataFrames.
df_dep = df_dep.rename(columns={
    "Ntop": "Ntop_dep"          # Effectif au niveau départemental
})

df_nat = df_nat.rename(columns={
    "Ntop": "Ntop_national",    # Effectif au niveau national
    "montant": "montant_national" # Montant de dépenses national (en euros)
})

In [ ]:
# --- Construction d'une clé métier composite ---
# Cette clé concatène les dimensions analytiques pour identifier de façon unique
# chaque combinaison (année × pathologie niv1 × niv2 × niv3 × top).
# Elle sert de clé de jointure entre les données départementales et nationales,
# en remplacement d'un merge multi-colonnes (plus lisible et performant via dict).

# Clé pour le DataFrame départemental
df_dep["key"] = (
    df_dep["annee"].astype(str) + "|" +
    df_dep["patho_niv1"].astype(str) + "|" +
    df_dep["patho_niv2"].astype(str) + "|" +
    df_dep["patho_niv3"].astype(str) + "|" +
    df_dep["top"].astype(str)
)

# Clé identique pour le DataFrame national (même structure de concaténation)
df_nat["key"] = (
    df_nat["annee"].astype(str) + "|" +
    df_nat["patho_niv1"].astype(str) + "|" +
    df_nat["patho_niv2"].astype(str) + "|" +
    df_nat["patho_niv3"].astype(str) + "|" +
    df_nat["top"].astype(str)
)

In [ ]:
# --- Calcul du poids du département dans l'effectif national ---
# Pour chaque groupe (année × pathologie × top), on calcule la part de patients
# pris en charge dans ce département par rapport au total national de ce groupe.
#
# Formule : weight = Ntop_dep_département / Σ(Ntop_dep) sur tous les départements du groupe
#
# Ce poids servira ensuite à ventiler les dépenses nationales au niveau départemental
# (hypothèse : les dépenses se répartissent proportionnellement aux effectifs).
df_dep["weight"] = df_dep["Ntop_dep"] / df_dep.groupby(
    ["annee", "patho_niv1", "patho_niv2", "patho_niv3", "top"]
)["Ntop_dep"].transform("sum")
# transform("sum") renvoie la somme du groupe pour chaque ligne → même index que df_dep,
# ce qui permet la division directe sans merge supplémentaire.

In [ ]:
# --- Création de dictionnaires de correspondance pour la jointure rapide ---
# On convertit les colonnes nationales en dict {key → valeur} pour un lookup en O(1).
# C'est plus efficace qu'un pd.merge sur de grands volumes.
ntop_nat_map = df_nat.set_index("key")["Ntop_national"].to_dict()
montant_nat_map = df_nat.set_index("key")["montant_national"].to_dict()

In [ ]:
# --- Jointure : enrichissement du DataFrame départemental avec les données nationales ---
# map() applique le dictionnaire comme une fonction de lookup :
# pour chaque ligne de df_dep, on récupère la valeur nationale correspondant à la même clé.
df_dep["Ntop_national"] = df_dep["key"].map(ntop_nat_map)
df_dep["montant_national"] = df_dep["key"].map(montant_nat_map)

In [ ]:
# --- Ventilation des dépenses nationales au niveau départemental ---
# Formule : montant_dep = weight × montant_national
#
# On applique le poids calculé précédemment pour estimer la part de dépenses
# imputable à ce département pour chaque combinaison (année × pathologie × top).
df_dep["montant_dep"] = df_dep["weight"] * df_dep["montant_national"]

In [ ]:
# --- Gestion des cas sans correspondance nationale (valeurs manquantes ou nulles) ---
# Si Ntop_national est NaN (clé absente du fichier national) ou égal à 0
# (aucun patient national recensé → division par zéro possible dans le poids),
# on force montant_dep à 0 pour éviter des valeurs aberrantes ou NaN en sortie.
df_dep.loc[
    (df_dep["Ntop_national"].isna()) |
    (df_dep["Ntop_national"] == 0),
    "montant_dep"
] = 0

In [ ]:
# --- Table de contrôle : vérification de la cohérence de la ventilation ---
# On ré-agrège les montants départementaux par groupe et on compare au montant national.
# L'écart doit être proche de 0 (tolérance numérique) si la ventilation est correcte.
# Un écart significatif indiquerait un problème dans les poids ou les données sources.
controle = (
    df_dep
    .groupby(
        ["annee", "patho_niv1", "patho_niv2", "patho_niv3", "top"],
        as_index=False
    )
    .agg(
        montant_dep_total=("montant_dep", "sum"),      # Somme des montants ventilés
        montant_national=("montant_national", "first") # Montant national de référence (identique pour le groupe)
    )
)

# Calcul de l'écart absolu entre la somme ventilée et le montant national
controle["ecart"] = (
    controle["montant_dep_total"] - controle["montant_national"]
)

df_dep.head()    # Aperçu du DataFrame enrichi après toutes les transformations

In [ ]:
df_dep.head()    # Aperçu du DataFrame enrichi après toutes les transformations

,annee,patho_niv1,patho_niv2,patho_niv3,top,montant_dep_total,montant_national,ecart
0,2015,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,Affections de longue durée (dont 31 et 32) pou...,ALD_CAT_CAT,"250,639,531",250639531,0
1,2015,Cancers,Autres cancers,Autres cancers actifs,CAN_AUT_ACT,"279,754,370",279754370,0
2,2015,Cancers,Autres cancers,Autres cancers sous surveillance,CAN_AUT_SUR,"36,619,131",36619131,0
3,2015,Cancers,Cancer bronchopulmonaire,Cancer bronchopulmonaire actif,CAN_BPU_ACT,"758,044",758044,0
4,2015,Cancers,Cancer bronchopulmonaire,Cancer bronchopulmonaire sous surveillance,CAN_BPU_SUR,0,0,0


In [ ]:
region = 32  # Code de la région étudiée (ici : Hauts-de-France ou Occitanie selon le référentiel)

data = {}  # Dictionnaire pour stocker le total des dépenses par année

# Boucle sur toutes les années présentes dans les données, triées chronologiquement
for year in sorted(df_dep["annee"].unique()):
    # Filtrage sur la région et l'année, puis somme des dépenses départementales
    total = df_dep[
        (df_dep["region"] == region) &
        (df_dep["annee"] == year)
    ]["montant_dep"].sum()
    data[year] = total

# Affichage avec calcul du taux de croissance annuel (Year-over-Year = YoY)
for i, year in enumerate(sorted(data.keys())):
    if i == 0:
        # Première année : pas d'année précédente, donc YoY non calculable
        print(f"{year} : {data[year]:,.0f}".replace(",", " "), "YoY: NA")
    else:
        prev = data[sorted(data.keys())[i-1]]  # Montant de l'année précédente
        # Formule du taux de variation : (valeur_n - valeur_n-1) / valeur_n-1 × 100
        pct = (data[year] - prev) / prev * 100
        print(f"{year} : {data[year]:,.0f}".replace(",", " "), f"YoY: {pct:.2f}%")


2015 : 1 165 317 843 YoY: NA
2016 : 1 424 014 090 YoY: 22.20%
2017 : 1 147 577 877 YoY: -19.41%
2018 : 1 248 775 184 YoY: 8.82%
2019 : 4 790 885 515 YoY: 283.65%
2020 : 5 156 946 564 YoY: 7.64%
2021 : 2 216 712 546 YoY: -57.02%
2022 : 2 195 202 680 YoY: -0.97%
2023 : 1 741 667 768 YoY: -20.66%


In [ ]:
# --- Extraction des données 2023 pour la région étudiée ---
# Sous-ensemble utilisé pour les analyses détaillées ci-dessous
df_filtre = df_dep[
    (df_dep["region"] == region) &
    (df_dep["annee"] == 2023)
]
df_filtre.head()

,annee,patho_niv1,patho_niv2,patho_niv3,top,cla_age_5,sexe,region,dept,Ntop_dep,...,prev,Niveau prioritaire,libelle_classe_age,libelle_sexe,tri,key,weight,Ntop_national,montant_national,montant_dep
807501,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,59,NaN,...,NaN,3,plus de 95 ans,tous sexes,78,2023|Maladies inflammatoires ou rares ou infec...,NaN,"9,650",261518,NaN
807504,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,60,NaN,...,NaN,3,plus de 95 ans,tous sexes,78,2023|Maladies inflammatoires ou rares ou infec...,NaN,"9,650",261518,NaN
807507,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,62,NaN,...,NaN,3,plus de 95 ans,tous sexes,78,2023|Maladies inflammatoires ou rares ou infec...,NaN,"9,650",261518,NaN
807510,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,80,NaN,...,NaN,3,plus de 95 ans,tous sexes,78,2023|Maladies inflammatoires ou rares ou infec...,NaN,"9,650",261518,NaN
807513,2023,Maladies inflammatoires ou rares ou infection VIH,Maladies rares,Mucoviscidose,RAR_MUC_IND,95et+,9,32,999,NaN,...,NaN,3,plus de 95 ans,tous sexes,78,2023|Maladies inflammatoires ou rares ou infec...,NaN,"9,650",261518,NaN


In [ ]:
# --- Agrégation des dépenses par pathologie de niveau 1 (catégorie principale) ---
# Permet d'identifier les grandes familles de maladies les plus coûteuses dans la région
result = (
    df_filtre
    .groupby("patho_niv1", as_index=False)["montant_dep"]
    .sum()
)
result.head()

,patho_niv1,montant_dep
0,Affections de longue durée (dont 31 et 32) pou...,"18,962,309"
1,Cancers,"247,756,431"
2,Diabète,"14,480,595"
3,Hospitalisation pour Covid-19,"350,457"
4,Hospitalisations hors pathologies repérées (av...,"428,194,838"


In [ ]:
# --- Top 10 des pathologies les plus coûteuses en 2023 pour la région ---
# Tri décroissant sur le montant pour faire ressortir les pathologies dominantes
result_sorted = result.sort_values("montant_dep", ascending=False)

# Option d'affichage : format numérique sans notation scientifique (ex: 1 234 567)
pd.set_option('display.float_format', '{:,.0f}'.format)

print(result_sorted.head(10))  # Affichage du top 10

                                           patho_niv1  montant_dep
4   Hospitalisations hors pathologies repérées (av...  428,194,838
14                     Total consommants tous régimes  266,220,422
1                                             Cancers  247,756,431
8   Maladies inflammatoires ou rares ou infection VIH  185,392,052
13  Pas de pathologie repérée, traitement, materni...  100,706,318
16  Traitements du risque vasculaire (hors patholo...  100,615,114
6                     Maladies cardioneurovasculaires   92,894,924
12               Maternité (avec ou sans pathologies)   77,798,053
5             Insuffisance rénale chronique terminale   66,998,672
17        Traitements psychotropes (hors pathologies)   56,903,229


In [ ]:
# --- Tableau croisé : dépenses par classe d'âge × pathologie (région 32, 2023) ---
# pivot_table crée une matrice avec :
#   - en lignes : les classes d'âge quinquennales (cla_age_5)
#   - en colonnes : les pathologies de niveau 1
#   - en valeurs : la somme des dépenses estimées
# fill_value=0 : les combinaisons sans données sont remplacées par 0 (pas de NaN)
pivot = df_filtre.pivot_table(
    index="cla_age_5",       # Axe lignes : tranches d'âge (ex: 0-4, 5-9, ..., 85+)
    columns="patho_niv1",    # Axe colonnes : grandes catégories de pathologies
    values="montant_dep",    # Valeur agrégée
    aggfunc="sum",           # Fonction d'agrégation : somme des dépenses
    fill_value=0             # Remplace les cellules vides par 0
)
print(pivot)

patho_niv1  Affections de longue durée (dont 31 et 32) pour d'autres causes  \
cla_age_5                                                                     
00-04                                                 224,997                 
05-09                                                 272,595                 
10-14                                                 325,634                 
15-19                                                 352,531                 
20-24                                                 278,942                 
25-29                                                 227,868                 
30-34                                                 253,405                 
35-39                                                 317,625                 
40-44                                                 347,091                 
45-49                                                 371,721                 
50-54                                               

In [ ]:
# --- Tableau croisé : dépenses par sexe × pathologie (tous départements, toutes années) ---
# Même logique que le pivot précédent mais sur l'ensemble du DataFrame (pas de filtre région/année).
# Permet une analyse nationale de la répartition des dépenses selon le genre et la pathologie.
pivot = df_dep.pivot_table(
    index="sexe",            # Axe lignes : sexe du patient (ex: 1=homme, 2=femme, 9=tous)
    columns="patho_niv1",    # Axe colonnes : pathologies de niveau 1
    values="montant_dep",    # Valeur agrégée
    aggfunc="sum",           # Somme des dépenses estimées
    fill_value=0
)
print(pivot)

patho_niv1  Affections de longue durée (dont 31 et 32) pour d'autres causes  \
sexe                                                                          
1                                               2,228,893,031                 
2                                               2,858,402,118                 
9                                               5,086,743,416                 

patho_niv1        Cancers        Diabète  Hospitalisation pour Covid-19  \
sexe                                                                      
1           4,979,620,237  6,454,246,858                     18,278,333   
2           5,591,054,349  5,326,835,533                     17,019,440   
9          10,578,164,873 11,781,069,655                     35,395,779   

patho_niv1  Hospitalisations hors pathologies repérées (avec ou sans pathologies, traitements ou maternité)  \
sexe                                                                                                          
1     